# **Random Forest Algorithm**

Prerequisites: Decision Trees, Bagging (`01_bagging.ipynb`).

## **1. Introduction**

A **Random Forest** is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and prevent overfitting. It is widely used for both **classification** and **regression** tasks.
<br><br>

![Random Forest.png](../../images/random_forest.png)

- **Key Idea**: Aggregate predictions from multiple trees to make a final prediction.
- **Why Random?**
  1. **Random Subset of Features**: At each split, only a random subset of features is considered.
  2. **Random Sampling**: Data is sampled with replacement (bootstrapping) to train individual trees.

##
---

## **2. Key Mathematical Concepts**

### **2.1 Ensemble Learning**
Random Forest uses **Bagging** (Bootstrap Aggregating) to combine predictions from multiple models.

#### Bagging Formula:
$$
\hat{y} = \frac{1}{T} \sum_{t=1}^{T} f_t(x)
$$
- \(T\): Number of trees.
- $(f_t(x))$: Prediction from tree \(t\).

#### **1. What Random Forest adds on top of plain Bagging**

Bagging alone (bootstrap + average) already reduces variance via $\text{Var}=\rho\sigma^2+\frac{(1-\rho)\sigma^2}{B}$ (derived in `01_bagging.ipynb`). Random Forest additionally decorrelates the trees by, at **each split**, only considering a random subset of $m<d$ features (commonly $m=\sqrt d$ for classification, $m=d/3$ for regression).
This directly lowers $\rho$ in the formula above — the specific mechanism,
not just "it works better."

##### **Worked numerical example — decorrelation's effect on the variance floor**

Single tree variance $\sigma^2=4.0$. Plain bagging achieves $\rho=0.5$
(trees still correlated since they see all features). Random Forest's
feature subsetting lowers this to $\rho=0.2$. With $B=50$ trees:
- Bagging floor: $0.5(4.0)=2.0$ (plus a vanishing second term)
- Random Forest floor: $0.2(4.0)=0.8$

Even with unlimited trees, bagging can never beat variance 2.0 on this
data, while Random Forest's lower correlation allows it down to 0.8 —
**quantifying** why RF usually outperforms plain bagging, not just citing
it as folklore.

### **2.2 Feature Randomness**
For a dataset with \(F\) features, only \($\sqrt{F}$\) (for classification) or \(F/3\) (for regression) features are used for splitting at each node. This introduces diversity among trees.

### **2.3. Feature Importance (Gini/Mean Decrease Impurity) - derivation**

For a feature $j$, importance is the total decrease in impurity (weighted
by the number of samples reaching that node) summed over all splits on
feature $j$, across all trees, then averaged:
$$\text{Imp}(j) = \frac1B\sum_{b=1}^B \sum_{t\in T_b: \text{split on }j} \frac{n_t}{n}\big[\text{Impurity}(t) - \text{weighted avg. Impurity(children)}\big]$$
This is exactly the Information-Gain quantity from `10-Decision-Tree-Additions.md`,
accumulated across the whole forest rather than a single tree — worth
explicitly connecting for students who already did the entropy/Gini
hand-calculation there.

##
---

## **3. Comparison: Decision Tree vs Random Forest**


| **Aspect**              | **Decision Tree**                                      | **Random Forest**                                   |
|--------------------------|-------------------------------------------------------|---------------------------------------------------|
| **Overfitting**          | Prone to overfitting.                                 | Reduces overfitting by averaging multiple trees.  |
| **Accuracy**             | May have lower accuracy due to high variance.         | Generally higher accuracy.                        |
| **Interpretability**     | Easy to interpret and visualize.                      | Harder to interpret as it aggregates multiple trees. |
| **Training Speed**       | Faster to train.                                      | Slower due to training multiple trees.            |
| **Feature Importance**   | Provides direct feature importance.                  | Provides averaged feature importance.             |
| **Robustness**           | Sensitive to noisy data.                              | Robust to noise and outliers.                     |


##
---

## **4. Implementation**

### **Classification Example**

Dataset: Iris Classification

This example predicts the type of iris flower using a Random Forest Classifier.

```python
# Import libraries
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

# Load Iris Dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split into Training and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the Model
clf.fit(X_train, y_train)

# Predict and Evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Feature Importance
feature_importance = pd.Series(clf.feature_importances_, index=iris.feature_names)
print("Feature Importance:")
print(feature_importance.sort_values(ascending=False))
```

In [1]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

# Load Iris Dataset
iris = load_iris()
iris.feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

[Implement of Random Forest Algorithm](10%20-%20Implement%20Random%20Forest%20Algorithm.ipynb)
###
---

### **Regression Example**

Dataset: California Housing Prices

This example predicts house prices using a Random Forest Regressor.

```python
#Import libraries
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error


# Load California Housing Dataset
data = fetch_california_housing()
X, y = data.data, data.target

# Split into Training and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize Random Forest Regressor
regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the Model
regressor.fit(X_train, y_train)

# Predict and Evaluate
y_pred = regressor.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")
```

[Implement of Random Forest Algorithm](10%20-%20Implement%20Random%20Forest%20Algorithm.ipynb)
###
---

## **5. Visualization**

- Feature Importance Visualization

```python
import matplotlib.pyplot as plt

# Visualize Feature Importance
feature_importance.sort_values().plot(kind='barh', title="Feature Importance", figsize=(10, 6))
plt.show()
```

##
---

## **6. Mathematical Formulations**

### **6.1 Random Sampling**
For 'N' samples in the dataset, Random Forest creates multiple training datasets by sampling 'N' samples with replacement (bootstrapping).

### **6.2 Aggregation of Predictions**

#### Classification:

$$\hat{y}​=Mode(y1​,y2​,…,yT​)$$

  - Takes the majority vote of *T* trees.

#### Regression:

$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} y_t$$

  - Takes the average of predictions from *T* trees.

### **6.3 Out-of-Bag (OOB) Error**

Uses samples not included in the bootstrap to evaluate model performance:

$$OOB \, Error = \frac{1}{N} \sum_{i=1}^{N} L(y_i, \hat{y}_i^{OOB})$$

  - L: Loss function (e.g., accuracy or MSE).

Each bootstrap sample of size $n$ (drawn with replacement from $n$ points)
excludes, on average, a fraction of the original data:
$$P(\text{point } i \text{ NOT in one bootstrap sample}) = \left(1-\frac1n\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0.368$$
So about 36.8% of points are "out-of-bag" for any given tree — each point
is OOB for roughly 36.8% of all trees, giving a free validation set: predict
each point using only the trees for which it was OOB, and average. This is
provably similar to k-fold CV error without needing an explicit held-out
split — worth stating precisely rather than just "RF gives you OOB error
for free."

#### **Worked numerical example**
$n=50$ training points. Expected number of trees (out of, say, $B=200$)
for which a specific point is OOB: $200\times0.368\approx73.6$ trees — enough
for a stable OOB prediction for every point.

In [1]:
## verify the OOB fraction and demonstrate feature importance

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# OOB fraction check
n = 50
print((1 - 1/n)**n, np.exp(-1))   # both ~0.3642 / 0.3679 -- converges to 1/e

data = load_iris()
rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0).fit(data.data, data.target)
print("OOB accuracy:", rf.oob_score_)
print("Feature importances:", dict(zip(data.feature_names, rf.feature_importances_)))

0.36416968008711675 0.36787944117144233
OOB accuracy: 0.9466666666666667
Feature importances: {'sepal length (cm)': 0.0975945008871649, 'sepal width (cm)': 0.02960937133879797, 'petal length (cm)': 0.4358979544337184, 'petal width (cm)': 0.4368981733403187}


##
---

## **7. Advantages and Disadvantages**

**Advantages**

- Reduces overfitting compared to Decision Trees.

- Handles large datasets with higher accuracy.

- Robust to noise and missing data.

- Provides feature importance rankings.

**Disadvantages**

- Requires more computational resources than a single Decision Tree.

- Harder to interpret due to the ensemble nature.

- May not perform well with sparse data.

![Advantages & Disadvantages.png](../../images/random_forest_advantages.png)

##
---

## **8. Comparison Example**

```python
# Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

# Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Decision Tree Accuracy: {dt_accuracy:.2f}")
print(f"Random Forest Accuracy: {rf_accuracy:.2f}")
```

##
---

## **9. Conclusion**

Key Takeaways

- Random Forest is a more robust and accurate model compared to a single Decision Tree due to ensemble learning.

- It minimizes overfitting and improves generalization by combining predictions from multiple trees.

- Decision Trees are faster and easier to interpret but less accurate.

##
---

## **10. Extensions**

### Hyperparameter Tuning

Use GridSearchCV or RandomizedSearchCV to optimize hyperparameters like:

- Number of estimators (n_estimators).

- Maximum depth of trees (max_depth).

- Minimum samples split (min_samples_split).

### Ensemble Variants

- Gradient Boosting: Sequentially builds trees to minimize residual errors.

- XGBoost: An optimized version of Gradient Boosting.

```python
from sklearn.model_selection import GridSearchCV

params = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
```